# Sentinel-2 optical and texture workflow for tree-covered coffee detectability

This notebook is part of the reproducibility material associated with the manuscript:

**Detectability limits of tree-covered coffee at Sentinel resolution: asymmetric gains from optical texture**

## Purpose

This notebook extracts and evaluates a parsimonious Sentinel-2 optical feature set for distinguishing tree-covered coffee from forest formations in the Dominican Republic. It also includes an optional Sentinel-2 GLCM texture block used to support the texture-enhanced analysis reported in the manuscript.

## Scientific rationale

The Sentinel-2 workflow is organised around two complementary predictor groups:

1. **Optical baseline attributes** — median Sentinel-2 surface reflectance bands and selected vegetation indices that describe canopy greenness, red-edge response and NIR-weighted vegetation signal.
2. **Optional optical texture attributes** — GLCM metrics derived from selected Sentinel-2 layers to represent local spatial contrast and heterogeneity at Sentinel resolution.

The notebook intentionally excludes DEM/topographic variables so that the Sentinel-2 contribution can be evaluated independently from terrain effects.


## Notebook organisation

The workflow is organised into the following blocks:

1. Environment configuration and Google Earth Engine initialisation;
2. Reading and standardising the reference polygon dataset;
3. Sentinel-2 acquisition for the DJFM seasonal windows and cloud masking;
4. Optical baseline construction using median bands and selected vegetation indices;
5. Optional GLCM texture construction;
6. Polygon-based extraction of optical and texture attributes;
7. Univariate separability diagnostics;
8. Random Forest optical-only modelling with stratified cross-validation;
9. Shade-level error analysis and feature importance;
10. Export of derived tables;
11. Optional spectral-envelope figure data;
12. Reproducibility summary and AI-use statement.

All code cells are intended to be run sequentially. The optional GLCM texture block can be disabled if only the optical baseline is required.


## Data privacy and repository use

The reference polygons used in this workflow may contain sensitive spatial information, including precise locations of coffee farms or field-survey areas. For this reason, the original vector data should not be committed to a public GitHub repository unless redistribution has been explicitly authorised.

For public sharing, this repository should contain the workflow, documentation and, when appropriate, anonymised or derived tabular outputs that do not expose private coordinates or field boundaries.


## Block 1 — Environment configuration and Google Earth Engine initialisation

This block installs and imports the Python packages required for the workflow, authenticates Google Earth Engine and defines user-specific paths.

Before running the notebook, update the configuration variables in the code cell below. Personal paths, cloud project identifiers and private filenames should not be committed to a public repository.


In [ ]:
# ============================================================
# Environment configuration
# ============================================================

# Install dependencies when running in Google Colab.
# In a local environment, these packages can also be installed from requirements.txt.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install earthengine-api geemap geopandas

# Core imports
import os
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu, kruskal
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Optional: mount Google Drive when running in Colab.
# Set to False if the input vector file is stored elsewhere.
USE_GOOGLE_DRIVE = True

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# ============================================================
# User configuration
# ============================================================

# Google Earth Engine project.
# For public repositories, avoid hard-coding personal or institutional project IDs.
# Use None for default initialisation, or set your project ID locally before running.
GEE_PROJECT_ID = None  # Example for local use only: "your-gee-project-id"

# Path to the local/private reference polygon dataset.
# This file is intentionally not included in the public GitHub repository.
INPUT_VECTOR_PATH = "/content/drive/MyDrive/path/to/reference_polygons.shp"

# Google Drive folder used for Earth Engine CSV exports.
DRIVE_EXPORT_FOLDER = "sentinel_coffee_outputs"

# ============================================================
# Earth Engine authentication and initialisation
# ============================================================

ee.Authenticate()

if GEE_PROJECT_ID is None:
    ee.Initialize()
else:
    ee.Initialize(project=GEE_PROJECT_ID)

print("Environment configured successfully.")


## Block 2 — Reference polygons and coffee shade groups

This block reads the reference polygon dataset, standardises the class labels and creates the grouping variable used to analyse coffee under different visual shade conditions.

The geometry is used only to extract satellite attributes. For public sharing, the polygon file itself should remain outside the repository unless it has been explicitly authorised for redistribution and does not expose sensitive field locations.


In [ ]:
# ============================================================
# Read and standardise the reference polygon dataset
# ============================================================

if not os.path.exists(INPUT_VECTOR_PATH):
    raise FileNotFoundError(
        "The reference polygon file was not found. "
        "Update INPUT_VECTOR_PATH in Block 1 before running this notebook."
    )

gdf = gpd.read_file(INPUT_VECTOR_PATH)

# Ensure CRS is EPSG:4326 for compatibility with Earth Engine.
if gdf.crs is None:
    print("CRS not defined. Setting as EPSG:4326 (assumed).")
    gdf = gdf.set_crs("EPSG:4326")

if str(gdf.crs).upper() != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Required input columns:
# - Class: thematic class label, expected to include "coffee" and "forest";
# - New_ID: shade-level code for coffee polygons.
required_input_columns = ["Class", "New_ID"]
missing_input_columns = [col for col in required_input_columns if col not in gdf.columns]
if missing_input_columns:
    raise ValueError(
        "Missing required input columns: "
        f"{missing_input_columns}. Expected at least: {required_input_columns}"
    )

# Standardise class and shade-level fields.
gdf["Class"] = gdf["Class"].astype(str).str.strip().str.lower()
gdf["New_ID"] = pd.to_numeric(gdf["New_ID"], errors="coerce")

# Stable sample ID based on row order.
# This avoids exposing original field identifiers in derived tables.
gdf = gdf.reset_index(drop=True)
gdf["sample_id"] = [f"id_{i:03d}" for i in range(len(gdf))]

print("Polygons loaded:", gdf.shape)

print("\nClass distribution:")
print(gdf["Class"].value_counts(dropna=False))

print("\nNew_ID distribution:")
print(gdf["New_ID"].value_counts(dropna=False).sort_index())

# Convert GeoDataFrame to Earth Engine FeatureCollection.
def gdf_to_fc(gdf):
    features = []

    for _, row in gdf.iterrows():
        geom = ee.Geometry(row.geometry.__geo_interface__)

        new_id = row["New_ID"]
        new_id = -9999 if pd.isna(new_id) else int(new_id)

        features.append(
            ee.Feature(
                geom,
                {
                    "class": row["Class"],
                    "sample_id": row["sample_id"],
                    "New_ID": new_id
                }
            )
        )

    return ee.FeatureCollection(features)

samples_fc = gdf_to_fc(gdf)
coffee_fc = samples_fc.filter(ee.Filter.eq("class", "coffee"))
forest_fc = samples_fc.filter(ee.Filter.eq("class", "forest"))

print("\nTotal sample polygons:", samples_fc.size().getInfo())
print("Coffee polygons:", coffee_fc.size().getInfo())
print("Forest polygons:", forest_fc.size().getInfo())
print("Coffee New_ID 1:", coffee_fc.filter(ee.Filter.eq("New_ID", 1)).size().getInfo())
print("Coffee New_ID 2:", coffee_fc.filter(ee.Filter.eq("New_ID", 2)).size().getInfo())
print("Coffee New_ID 3:", coffee_fc.filter(ee.Filter.eq("New_ID", 3)).size().getInfo())


## Block 3 — Sentinel-2 DJFM collection and cloud masking

In [ ]:
# Sentinel-2 bands retained for the optical baseline
S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A"]

# DJFM windows. These match the seasonal window used for Sentinel-1.
SEASONS = [
    ("2021-12-01", "2022-03-31"),
    ("2022-12-01", "2023-03-31"),
    ("2023-12-01", "2024-03-31"),
    ("2024-12-01", "2025-03-31"),
    ("2025-12-01", "2026-03-31"),
]

# Compact ROI based on sample extent
roi = samples_fc.geometry().bounds().buffer(3000)

def get_s2_raw(seasons, roi_geom):
    col = ee.ImageCollection([])
    for start_date, end_date in seasons:
        season_col = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(roi_geom)
            .filterDate(start_date, end_date)
        )
        col = col.merge(season_col)
    return col

s2_raw = get_s2_raw(SEASONS, roi)
print("S2 images raw:", s2_raw.size().getInfo())
print("Mean CLOUDY_PIXEL_PERCENTAGE:", ee.Number(s2_raw.aggregate_mean("CLOUDY_PIXEL_PERCENTAGE")).getInfo())

def mask_s2_scl(image):
    scl = image.select("SCL")

    # Remove cloud shadow, medium/high cloud probability, cirrus and snow/ice.
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return (
        image.updateMask(mask)
        .select(S2_BANDS)
        .multiply(0.0001)
        .copyProperties(image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])
    )

s2 = s2_raw.map(mask_s2_scl)
print("S2 images after SCL mask:", s2.size().getInfo())

## Block 4 — Build optical baseline image: median bands + selected indices

The optical baseline is intentionally parsimonious. It keeps all median Sentinel-2 bands and only four indices with distinct roles:

- **NDVI**: general vegetation cover/vigour;
- **NDRE**: red-edge response in dense vegetation;
- **MCARI2**: chlorophyll/canopy structure with partial soil/background compensation;
- **NIRv**: NIR-weighted vegetation signal.

CIre, EVI2 and MSAVI2 are not included in the main model to reduce redundancy.

In [ ]:
EPS = 1e-6

def safe_div(n, d):
    return n.divide(d.abs().add(EPS))

def ndvi(red, nir):
    return safe_div(nir.subtract(red), nir.add(red))

def ndre(rededge, nir):
    return safe_div(nir.subtract(rededge), nir.add(rededge))

def nirv(red, nir):
    return ndvi(red, nir).multiply(nir)

def mcari2(green, red, rededge, nir):
    num = nir.subtract(red).multiply(2.5).subtract(
        nir.subtract(green).multiply(1.3)
    )
    denom_inside = (
        (nir.multiply(2).add(1)).pow(2)
        .subtract(nir.multiply(6))
        .add(red.sqrt().multiply(5))
        .subtract(0.5)
    )
    denom = denom_inside.max(EPS).sqrt()
    return num.multiply(1.5).divide(denom)

# Median reflectance image
img_median = s2.reduce(ee.Reducer.median()).clip(roi)

# Select median bands
B2 = img_median.select("B2_median")
B3 = img_median.select("B3_median")
B4 = img_median.select("B4_median")
B5 = img_median.select("B5_median")
B6 = img_median.select("B6_median")
B7 = img_median.select("B7_median")
B8 = img_median.select("B8_median")
B8A = img_median.select("B8A_median")

# Selected indices
index_img = ee.Image.cat([
    ndvi(B4, B8).rename("NDVI_median"),
    ndre(B5, B8A).rename("NDRE_median"),
    mcari2(B3, B4, B5, B8).rename("MCARI2_median"),
    nirv(B4, B8).rename("NIRv_median")
]).clip(roi)

# Final optical baseline image
feature_img_optical = img_median.addBands(index_img)

OPTICAL_FEATURES = [
    "B2_median", "B3_median", "B4_median", "B5_median",
    "B6_median", "B7_median", "B8_median", "B8A_median",
    "NDVI_median", "NDRE_median", "MCARI2_median", "NIRv_median"
]

# Keep only intended features, avoiding accidental additional bands.
feature_img_optical = feature_img_optical.select(OPTICAL_FEATURES)

print("Optical baseline band count:", feature_img_optical.bandNames().size().getInfo())
print("Optical baseline bands:", feature_img_optical.bandNames().getInfo())

## Block 5 — Optional GLCM texture image for sensitivity analysis

This block derives a minimal GLCM texture set from two Sentinel-2 layers:

- **B4_median**, representing red reflectance/shadow/canopy-background contrast;
- **NDRE_median**, representing red-edge canopy response.

These features should be used only for the complementary texture sensitivity analysis, not for the main Optical baseline unless explicitly decided later.

In [ ]:
# Toggle this flag if you do not want to compute GLCM texture.
COMPUTE_S2_GLCM = True

if COMPUTE_S2_GLCM:
    def to_byte_unit(img, min_val, max_val, name):
        return (
            img.unitScale(min_val, max_val)
            .clamp(0, 1)
            .multiply(255)
            .toByte()
            .rename(name)
        )

    # Quantise input layers for GLCM.
    # Reflectance and index ranges are intentionally broad and conservative.
    b4_byte = to_byte_unit(feature_img_optical.select("B4_median"), 0.0, 0.35, "B4_tex")
    ndre_byte = to_byte_unit(feature_img_optical.select("NDRE_median"), -0.2, 0.8, "NDRE_tex")

    glcm_input = ee.Image.cat([b4_byte, ndre_byte])
    glcm_raw = glcm_input.glcmTexture(size=3)

    # Earth Engine GLCM suffixes: _contrast and _ent are the two retained metrics.
    s2_glcm_img = ee.Image.cat([
        glcm_raw.select("B4_tex_contrast").rename("B4_glcm_contrast"),
        glcm_raw.select("B4_tex_ent").rename("B4_glcm_entropy"),
        glcm_raw.select("NDRE_tex_contrast").rename("NDRE_glcm_contrast"),
        glcm_raw.select("NDRE_tex_ent").rename("NDRE_glcm_entropy")
    ]).clip(roi)

    S2_GLCM_FEATURES = [
        "B4_glcm_contrast", "B4_glcm_entropy",
        "NDRE_glcm_contrast", "NDRE_glcm_entropy"
    ]

    print("S2 GLCM bands:", s2_glcm_img.bandNames().getInfo())
else:
    s2_glcm_img = None
    S2_GLCM_FEATURES = []
    print("S2 GLCM computation skipped.")

## Block 6 — Polygon-level extraction helpers

In [ ]:
SCALE_M = 10
SPATIAL_REDUCER = ee.Reducer.mean()

def fc_to_df(fc):
    info = fc.getInfo()
    return pd.json_normalize([f["properties"] for f in info["features"]])

def assign_group(row):
    if row["class"] == "forest":
        return "forest"
    elif row["class"] == "coffee" and row["shade_level"] == 1:
        return "coffee_low_shade"
    elif row["class"] == "coffee" and row["shade_level"] == 2:
        return "coffee_intermediate_shade"
    elif row["class"] == "coffee" and row["shade_level"] == 3:
        return "coffee_high_shade"
    else:
        return "unknown"

def extract_samples(image, collection, reducer=SPATIAL_REDUCER, scale=SCALE_M):
    out_fc = image.reduceRegions(
        collection=collection,
        reducer=reducer,
        scale=scale,
        tileScale=4
    )
    df = fc_to_df(out_fc)

    if "class" not in df.columns:
        raise ValueError("class column not found after extraction.")
    if "sample_id" not in df.columns:
        raise ValueError("sample_id column not found after extraction.")
    if "New_ID" not in df.columns:
        raise ValueError("New_ID column not found after extraction.")

    df["class"] = df["class"].astype(str).str.strip().str.lower()
    df["New_ID"] = pd.to_numeric(df["New_ID"], errors="coerce")
    df["shade_level"] = df["New_ID"].copy()
    df.loc[df["class"] == "forest", "shade_level"] = np.nan
    df["group"] = df.apply(assign_group, axis=1)

    return df.sort_values(["class", "sample_id"]).reset_index(drop=True)

## Block 7 — Extract optical baseline table

In [ ]:
all_optical_df = extract_samples(
    image=feature_img_optical,
    collection=samples_fc,
    reducer=SPATIAL_REDUCER,
    scale=SCALE_M
)

print("Rows:", all_optical_df.shape[0])
print("Columns:", all_optical_df.shape[1])
print("Class distribution:")
print(all_optical_df["class"].value_counts(dropna=False))
print("Group distribution:")
print(all_optical_df["group"].value_counts(dropna=False))
print("Missing values in optical features:")
display(all_optical_df[OPTICAL_FEATURES].isna().sum().sort_values(ascending=False))

display(all_optical_df.head())

## Block 8 — Extract optional S2 GLCM texture table

In [ ]:
if COMPUTE_S2_GLCM:
    all_s2_glcm_df = extract_samples(
        image=s2_glcm_img,
        collection=samples_fc,
        reducer=SPATIAL_REDUCER,
        scale=SCALE_M
    )

    # Keep metadata + texture columns
    meta_cols = ["class", "sample_id", "New_ID", "shade_level", "group"]
    all_s2_glcm_df = all_s2_glcm_df[meta_cols + S2_GLCM_FEATURES].copy()

    print("Rows:", all_s2_glcm_df.shape[0])
    print("Columns:", all_s2_glcm_df.shape[1])
    print("Missing values in S2 GLCM features:")
    display(all_s2_glcm_df[S2_GLCM_FEATURES].isna().sum().sort_values(ascending=False))
    display(all_s2_glcm_df.head())
else:
    all_s2_glcm_df = None

## Block 9 — Binary separability diagnostics for optical baseline

In [ ]:
def cohens_d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    pooled_sd = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd if pooled_sd != 0 else np.nan

def classify_auc(v):
    if v >= 0.80:
        return "strong"
    elif v >= 0.70:
        return "moderate"
    elif v >= 0.60:
        return "weak"
    else:
        return "very weak"

def classify_d(v):
    av = abs(v)
    if av >= 0.80:
        return "large"
    elif av >= 0.50:
        return "medium"
    elif av >= 0.20:
        return "small"
    else:
        return "very small"

def binary_separability(df, feature_cols):
    coffee = df[df["class"] == "coffee"].copy()
    forest = df[df["class"] == "forest"].copy()

    rows = []
    for col in feature_cols:
        x = coffee[col].dropna()
        y = forest[col].dropna()
        if len(x) == 0 or len(y) == 0:
            continue
        y_true = np.concatenate([np.ones(len(x)), np.zeros(len(y))])
        scores = np.concatenate([x.values, y.values])
        auc_raw = roc_auc_score(y_true, scores)
        auc = max(auc_raw, 1 - auc_raw)
        rows.append({
            "Feature": col,
            "AUC": auc,
            "AUC_raw": auc_raw,
            "Cohens_d": cohens_d(x, y),
            "abs_Cohens_d": abs(cohens_d(x, y)),
            "p_value": mannwhitneyu(x, y, alternative="two-sided").pvalue,
            "Mean_coffee": x.mean(),
            "Mean_forest": y.mean(),
            "Diff_forest_minus_coffee": y.mean() - x.mean(),
            "n_coffee": len(x),
            "n_forest": len(y)
        })
    out = pd.DataFrame(rows).sort_values(["AUC", "abs_Cohens_d"], ascending=False).reset_index(drop=True)
    out["AUC_class"] = out["AUC"].apply(classify_auc)
    out["Effect_class"] = out["Cohens_d"].apply(classify_d)
    return out

optical_sep_df = binary_separability(all_optical_df, OPTICAL_FEATURES)
display(optical_sep_df)

if COMPUTE_S2_GLCM:
    s2_glcm_sep_df = binary_separability(all_s2_glcm_df, S2_GLCM_FEATURES)
    print("S2 GLCM separability diagnostics:")
    display(s2_glcm_sep_df)

## Block 10 — Optical Random Forest with 5-fold stratified CV

In [ ]:
df_model_optical = all_optical_df[all_optical_df["class"].isin(["coffee", "forest"])].copy()
df_model_optical = df_model_optical.dropna(subset=OPTICAL_FEATURES).copy()
df_model_optical["target"] = df_model_optical["class"].map({"forest": 0, "coffee": 1})

X_optical = df_model_optical[OPTICAL_FEATURES]
y_optical = df_model_optical["target"]

rf_optical = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_optical = cross_val_predict(
    rf_optical, X_optical, y_optical,
    cv=cv, method="predict", n_jobs=-1
)

y_prob_optical = cross_val_predict(
    rf_optical, X_optical, y_optical,
    cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

optical_cv_results = df_model_optical.copy()
optical_cv_results["y_true"] = y_optical.values
optical_cv_results["y_pred"] = y_pred_optical
optical_cv_results["y_prob_coffee"] = y_prob_optical
optical_cv_results["correct"] = optical_cv_results["y_true"] == optical_cv_results["y_pred"]
optical_cv_results["predicted_class"] = optical_cv_results["y_pred"].map({0: "forest", 1: "coffee"})

optical_metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC", "Average precision"],
    "Value": [
        accuracy_score(y_optical, y_pred_optical),
        precision_score(y_optical, y_pred_optical),
        recall_score(y_optical, y_pred_optical),
        f1_score(y_optical, y_pred_optical),
        roc_auc_score(y_optical, y_prob_optical),
        average_precision_score(y_optical, y_prob_optical)
    ]
})

display(optical_metrics_df)
print("Classification report:")
print(classification_report(y_optical, y_pred_optical, target_names=["Forest", "Coffee"], digits=4))

cm_optical = pd.DataFrame(
    confusion_matrix(y_optical, y_pred_optical),
    index=["True Forest", "True Coffee"],
    columns=["Predicted Forest", "Predicted Coffee"]
)
display(cm_optical)

## Block 11 — Optical error analysis by coffee shade level

In [ ]:
coffee_optical_cv = optical_cv_results[optical_cv_results["class"] == "coffee"].copy()

shade_error_optical_summary = (
    coffee_optical_cv
    .groupby("group")
    .agg(
        n=("sample_id", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        mean_prob_coffee=("y_prob_coffee", "mean"),
        sd_prob_coffee=("y_prob_coffee", "std"),
        min_prob_coffee=("y_prob_coffee", "min"),
        max_prob_coffee=("y_prob_coffee", "max")
    )
    .reset_index()
)

shade_error_optical_summary["misclassified"] = shade_error_optical_summary["n"] - shade_error_optical_summary["correct"]
shade_error_optical_summary["error_rate"] = shade_error_optical_summary["misclassified"] / shade_error_optical_summary["n"]

shade_order = ["coffee_low_shade", "coffee_intermediate_shade", "coffee_high_shade"]
shade_error_optical_summary["group"] = pd.Categorical(
    shade_error_optical_summary["group"],
    categories=shade_order,
    ordered=True
)
shade_error_optical_summary = shade_error_optical_summary.sort_values("group")

display(shade_error_optical_summary)

## Block 12 — Optical feature importance

In [ ]:
rf_optical_final = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf_optical_final.fit(X_optical, y_optical)

optical_importance_df = pd.DataFrame({
    "Feature": OPTICAL_FEATURES,
    "Importance": rf_optical_final.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(optical_importance_df)

## Block 13 — Export optical baseline and optional GLCM tables to Drive

In [ ]:
# Export optical baseline table
optical_fc = feature_img_optical.reduceRegions(
    collection=samples_fc,
    reducer=SPATIAL_REDUCER,
    scale=SCALE_M,
    tileScale=4
)

task_optical = ee.batch.Export.table.toDrive(
    collection=optical_fc,
    description="S2_optical_attributes_reduced_DJFM_3class",
    folder=DRIVE_EXPORT_FOLDER,
    fileNamePrefix="S2_optical_attributes_reduced_DJFM_3class",
    fileFormat="CSV"
)
task_optical.start()
print("Export started: S2_optical_attributes_reduced_DJFM_3class.csv")

# Optional export of S2 GLCM table for complementary sensitivity analysis
if COMPUTE_S2_GLCM:
    s2_glcm_fc = s2_glcm_img.reduceRegions(
        collection=samples_fc,
        reducer=SPATIAL_REDUCER,
        scale=SCALE_M,
        tileScale=4
    )

    task_s2_glcm = ee.batch.Export.table.toDrive(
        collection=s2_glcm_fc,
        description="S2_GLCM_texture_DJFM_3class_optional",
        folder=DRIVE_EXPORT_FOLDER,
        fileNamePrefix="S2_GLCM_texture_DJFM_3class_optional",
        fileFormat="CSV"
    )
    task_s2_glcm.start()
    print("Export started: S2_GLCM_texture_DJFM_3class_optional.csv")

## Block 14 — Optional spectral envelope figure data

This block is for figure preparation only. It is not part of the modelling feature set.

In [ ]:
img_p10 = s2.reduce(ee.Reducer.percentile([10])).clip(roi)
img_p90 = s2.reduce(ee.Reducer.percentile([90])).clip(roi)

envelope_img = img_median.addBands(img_p10).addBands(img_p90).clip(roi)

envelope_df = extract_samples(
    image=envelope_img,
    collection=samples_fc,
    reducer=SPATIAL_REDUCER,
    scale=SCALE_M
)

print("Envelope rows:", envelope_df.shape[0])
print("Envelope columns:", envelope_df.shape[1])
print([c for c in envelope_df.columns if "_p10" in c or "_p90" in c][:20])
display(envelope_df.head())

In [ ]:
# ============================================================
# FINAL SUMMARY BLOCK — SENTINEL-2 OPTICAL
# Reproducibility summary for project records and manuscript reporting
# ============================================================

import pandas as pd
import numpy as np

print("\n" + "="*70)
print("OPTICAL MODEL — FINAL SUMMARY")
print("="*70)

# ------------------------------------------------------------
# 1. Metrics
# ------------------------------------------------------------
print("\nOPTICAL MODEL:")
try:
    metric_map = dict(zip(optical_metrics_df["Metric"], optical_metrics_df["Value"]))
    print(f"Accuracy = {metric_map.get('Accuracy', np.nan):.4f}")
    print(f"Precision = {metric_map.get('Precision', np.nan):.4f}")
    print(f"Recall = {metric_map.get('Recall', np.nan):.4f}")
    print(f"F1-score = {metric_map.get('F1-score', np.nan):.4f}")
    print(f"ROC AUC = {metric_map.get('ROC AUC', np.nan):.4f}")
    print(f"Average precision = {metric_map.get('Average precision', np.nan):.4f}")
except Exception as e:
    print("[ERROR] Could not print Optical metrics.")
    print(e)

# ------------------------------------------------------------
# 2. Confusion matrix
# ------------------------------------------------------------
print("\nCONFUSION MATRIX:")
try:
    print(f"True Forest / Predicted Forest = {cm_optical.loc['True Forest', 'Predicted Forest']}")
    print(f"True Forest / Predicted Coffee = {cm_optical.loc['True Forest', 'Predicted Coffee']}")
    print(f"True Coffee / Predicted Forest = {cm_optical.loc['True Coffee', 'Predicted Forest']}")
    print(f"True Coffee / Predicted Coffee = {cm_optical.loc['True Coffee', 'Predicted Coffee']}")
except Exception as e:
    print("[ERROR] Could not print Optical confusion matrix.")
    print(e)

# ------------------------------------------------------------
# 3. Feature importance
# ------------------------------------------------------------
print("\nOPTICAL FEATURE IMPORTANCE:")
try:
    display(optical_importance_df)
    for _, row in optical_importance_df.iterrows():
        print(f"{row['Feature']} = {row['Importance']:.6f}")
except Exception as e:
    print("[ERROR] Could not print Optical feature importance.")
    print(e)

# ------------------------------------------------------------
# 4. Binary separability diagnostics
# ------------------------------------------------------------
print("\nOPTICAL BINARY SEPARABILITY — COFFEE VS FOREST:")
try:
    display(optical_sep_df)
except Exception as e:
    print("[WARNING] optical_sep_df not available.")
    print(e)

# ------------------------------------------------------------
# 5. Error summary by true/predicted direction
# ------------------------------------------------------------
print("\nOPTICAL ERROR DIRECTION SUMMARY:")
try:
    optical_error_direction_df = optical_cv_results.copy()

    def classify_error_direction(row):
        if row["correct"]:
            return "Correct"
        elif row["class"] == "coffee" and row["predicted_class"] == "forest":
            return "Coffee_to_Forest"
        elif row["class"] == "forest" and row["predicted_class"] == "coffee":
            return "Forest_to_Coffee"
        else:
            return "Other"

    optical_error_direction_df["error_direction"] = optical_error_direction_df.apply(
        classify_error_direction, axis=1
    )

    optical_error_summary = (
        optical_error_direction_df
        .groupby("error_direction")
        .agg(
            n=("sample_id", "count"),
            mean_prob_coffee=("y_prob_coffee", "mean"),
            sd_prob_coffee=("y_prob_coffee", "std")
        )
        .reset_index()
        .sort_values("error_direction")
    )

    display(optical_error_summary)
except Exception as e:
    print("[ERROR] Could not print Optical error direction summary.")
    print(e)

# ------------------------------------------------------------
# 6. Shade-level error summary
# ------------------------------------------------------------
print("\nOPTICAL SHADE-LEVEL ERROR SUMMARY:")
try:
    coffee_optical_cv = optical_cv_results[
        optical_cv_results["class"] == "coffee"
    ].copy()

    shade_error_optical_summary = (
        coffee_optical_cv
        .groupby("group")
        .agg(
            n=("sample_id", "count"),
            correct=("correct", "sum"),
            accuracy=("correct", "mean"),
            mean_prob_coffee=("y_prob_coffee", "mean"),
            sd_prob_coffee=("y_prob_coffee", "std"),
            min_prob_coffee=("y_prob_coffee", "min"),
            max_prob_coffee=("y_prob_coffee", "max")
        )
        .reset_index()
    )

    shade_error_optical_summary["misclassified"] = (
        shade_error_optical_summary["n"] - shade_error_optical_summary["correct"]
    )

    shade_error_optical_summary["error_rate"] = (
        shade_error_optical_summary["misclassified"] / shade_error_optical_summary["n"]
    )

    shade_order = [
        "coffee_low_shade",
        "coffee_intermediate_shade",
        "coffee_high_shade"
    ]

    shade_error_optical_summary["group"] = pd.Categorical(
        shade_error_optical_summary["group"],
        categories=shade_order,
        ordered=True
    )

    shade_error_optical_summary = shade_error_optical_summary.sort_values("group")

    display(shade_error_optical_summary)

    for _, row in shade_error_optical_summary.iterrows():
        print(
            f"{row['group']}: "
            f"n={int(row['n'])}, "
            f"correct={int(row['correct'])}, "
            f"misclassified={int(row['misclassified'])}, "
            f"accuracy={row['accuracy']:.4f}, "
            f"error_rate={row['error_rate']:.4f}, "
            f"mean_prob_coffee={row['mean_prob_coffee']:.4f}"
        )
except Exception as e:
    print("[WARNING] Could not print Optical shade-level error summary.")
    print(e)

# ------------------------------------------------------------
# 7. Optional GLCM diagnostics, if computed
# ------------------------------------------------------------
print("\nS2 GLCM OPTIONAL DIAGNOSTICS:")
try:
    if "s2_glcm_sep_df" in globals():
        display(s2_glcm_sep_df)
    else:
        print("S2 GLCM separability table was not computed or is not available.")
except Exception as e:
    print("[WARNING] Could not print S2 GLCM diagnostics.")
    print(e)

# ------------------------------------------------------------
# 8. List of variables used
# ------------------------------------------------------------
print("\nOPTICAL FEATURES USED IN THE MODEL:")
try:
    for f in OPTICAL_FEATURES:
        print(f"- {f}")
except Exception as e:
    print("[ERROR] Could not print Optical feature list.")
    print(e)

print("\n" + "="*70)
print("END OF OPTICAL SUMMARY")
print("="*70)

## AI-use statement

Large language models were used as writing and coding support tools during the preparation of this notebook, specifically to assist with code organisation, documentation, readability and repository formatting. All scientific decisions, dataset interpretation, parameter choices and final analytical responsibility remain with the authors.
